# MDD Analysis — SpeakUP Matching API Latency

**Метрика:** P95 Response Latency (секунды)  
**Метод:** Welch t-test (независимые выборки)  
**Решение:** Принять/отклонить ADR-001 (FAISS ANN + Redis cache)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

# === Данные из задания ===
np.random.seed(42)
existing_system_responses = np.random.normal(loc=3.5, scale=0.4, size=500000)
improved_system_responses = np.random.normal(loc=2.0, scale=0.4, size=500000)

# === Визуализация (код из задания) ===
plt.figure(figsize=(10, 6))
sns.kdeplot(existing_system_responses, label='Существующая система', fill=True, color='red')
sns.kdeplot(improved_system_responses, label='Улучшенная система', fill=True, color='green')
plt.title('Сравнение времени отклика системы')
plt.xlabel('Время отклика (секунды)')
plt.ylabel('Наблюдения')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
ax = plt.gca()
ymin, ymax = ax.get_ylim()
num_ticks = 5
new_yticks = np.linspace(ymin, ymax, num_ticks)
new_yticklabels = [f'{int((tick / ymax) * 100)}%' if ymax != 0 else '0%' for tick in new_yticks]
ax.set_yticks(new_yticks)
ax.set_yticklabels(new_yticklabels)
plt.savefig('latency_comparison.png', dpi=150)
plt.show()
print('Chart saved')

## Welch t-test

**H₀:** mean_improved ≥ mean_existing (нет улучшения)  
**H₁:** mean_improved < mean_existing (улучшение)  
α = 0.05, одностороннее

In [ ]:
print('=== Дескриптивная статистика ===')
print(f'Существующая: mean={existing_system_responses.mean():.3f}с, P95={np.percentile(existing_system_responses,95):.3f}с')
print(f'Улучшенная:   mean={improved_system_responses.mean():.3f}с, P95={np.percentile(improved_system_responses,95):.3f}с')

np.random.seed(0)
s_exist = np.random.choice(existing_system_responses, 10000, replace=False)
s_impr  = np.random.choice(improved_system_responses, 10000, replace=False)

t_stat, p_two = stats.ttest_ind(s_impr, s_exist, equal_var=False)
p_one = p_two / 2
alpha = 0.05

print(f'\nt-statistic: {t_stat:.2f}')
print(f'p-value (one-sided): {p_one:.4e}')
print(f'alpha: {alpha}')

if t_stat < 0 and p_one < alpha:
    print(f'\n✅ H₀ ОТКЛОНЕНА: p={p_one:.2e} < α={alpha}')
    print('   Решение: ПРИНЯТЬ ADR-001 (FAISS ANN + Redis cache)')
else:
    print(f'\n❌ H₀ НЕ ОТКЛОНЕНА — изменение не обосновано')

In [ ]:
# Effect size
pooled_std = np.sqrt((s_exist.std()**2 + s_impr.std()**2) / 2)
cohens_d   = (s_exist.mean() - s_impr.mean()) / pooled_std
p95_e = np.percentile(existing_system_responses, 95)
p95_i = np.percentile(improved_system_responses, 95)

print(f"Cohen's d: {cohens_d:.2f} (>0.8 = large effect)")
print(f"P95 reduction: {p95_e:.2f}с → {p95_i:.2f}с  (-{(p95_e-p95_i)/p95_e*100:.1f}%)")
print(f"Mean reduction: {s_exist.mean():.2f}с → {s_impr.mean():.2f}с  (-{(s_exist.mean()-s_impr.mean())/s_exist.mean()*100:.1f}%)")

slo = 0.5
print(f"\nВ SLO 500мс (существующая): {(existing_system_responses < slo).mean()*100:.1f}%")
print(f"В SLO 500мс (улучшенная):   {(improved_system_responses < slo).mean()*100:.1f}%")
print('\n→ Следующая итерация v2: ONNX + FAISS → целевой P95 ≤ 500мс')

## Вывод

| Показатель | Существующая | Улучшенная | Δ |
|------------|-------------|-----------|---|
| Mean | 3.50 с | 2.00 с | **-42.9%** |
| P95 | 4.16 с | 2.66 с | **-36.1%** |
| t-test | — | t=-265.9, p≈0 | **H₀ отклонена** |
| Cohen's d | — | ~3.75 | **Large** |

**Решение:** ADR-001 принято. FAISS ANN + Redis cache внедряется в v2.